In [ ]:
import requests
import pandas as pd
import re
from difflib import SequenceMatcher

pd.set_option('display.max_colwidth', None)

def fetch_openalex_data(author_id):
    """Holt die rohen Daten von der API."""
    paper_daten = []
    page = 1
    while True:
        url = f"https://api.openalex.org/works?filter=author.id:{author_id}&per-page=200&page={page}"
        response = requests.get(url)
        if response.status_code != 200: break
        results = response.json().get('results', [])
        if not results: break 
        paper_daten.extend(results)
        page += 1
    return paper_daten

def extract_flat_data(paper_daten, author_id):
    """Extrahiert die Daten inkl. Journal, Domain UND Field."""
    stationen = []
    gute_typen = {'education', 'facility', 'government', 'company', 'archive'}

    for paper in paper_daten:
        jahr = paper.get('publication_year')
        datum = paper.get('publication_date') 
        titel = paper.get('title')
        work_type = paper.get('type', 'other') 
        
        # --- NEU: DATENMÜLL FILTERN ---
        if work_type == 'dataset': continue 
        if jahr is None or not titel: continue
        if not datum: datum = f"{int(jahr)}-01-01"
    
        primary_loc = paper.get('primary_location') or {}
        source = primary_loc.get('source') or {}
        journal_name = source.get('display_name') or 'Kein Journal angegeben'
        
        # --- NEU: WIR HOLEN BEIDES! (DOMAIN & FIELD) ---
        topic_info = paper.get('primary_topic') or {}
        
        domain_info = topic_info.get('domain') or {}
        paper_domain = domain_info.get('display_name', 'Unknown')
        
        field_info = topic_info.get('field') or {}
        paper_field = field_info.get('display_name', 'Unknown')
        
        if source.get('type') == 'repository' or 'ssrn' in str(journal_name).lower() or 'nber' in str(journal_name).lower():
            echter_typ = 'preprint'
        else:
            echter_typ = work_type

        api_link = paper.get('id', '').replace('https://openalex.org/', 'https://api.openalex.org/')

        for authorship in paper.get('authorships', []):
            author_info = authorship.get('author')
            if not author_info: continue
                
            auth_id = author_info.get('id')
            if auth_id and author_id in str(auth_id): 
                
                institutions = authorship.get('institutions') or []
                valid_unis = [inst.get('display_name') for inst in institutions if inst.get('type') in gute_typen]
                
                if not valid_unis:
                    raw = authorship.get('raw_affiliation_strings') or []
                    if raw: valid_unis = [f"[RAW] {raw[0]}"]
                
                for uni in valid_unis:
                    if uni: 
                        stationen.append({
                            'Jahr': int(jahr),
                            'Datum': datum,
                            'Uni': uni,
                            'Titel': titel,
                            'Work_Type': echter_typ,
                            'Journal': journal_name,
                            'Domain': paper_domain, # <--- BEIDES WIRD ÜBERGEBEN
                            'Field': paper_field,   # <--- BEIDES WIRD ÜBERGEBEN
                            'Link': api_link
                        })
                break 
    return pd.DataFrame(stationen)


def clean_title_for_comparison(title):
    if not isinstance(title, str): return ""
    clean = title.lower()
    clean = re.sub(r'[^a-z0-9]', '', clean)
    clean = clean.replace('s', '')
    return clean

def similar(a, b):
    return SequenceMatcher(None, a, b).ratio()

def deduplicate_papers(df):
    """
    Fuzzy Deduplizierung über alle Jahre hinweg.
    """
    if df.empty: return df

    df['Clean_Title'] = df['Titel'].apply(clean_title_for_comparison)
    
    unique_titles = df['Clean_Title'].unique()
    sorted_titles = sorted(unique_titles, key=len, reverse=True)
    
    master_mapping = {}
    
    for title in sorted_titles:
        matched_master = title 
        for potential_master in master_mapping.values():
            is_substring = (title in potential_master) and len(title) > 15
            is_similar = similar(title, potential_master) > 0.85
            if is_substring or is_similar:
                matched_master = potential_master
                break
        master_mapping[title] = matched_master

    df['Title_Group'] = df['Clean_Title'].map(master_mapping)

    def bestimme_prioritaet(row):
        journal_name = str(row['Journal']).lower()
        if 'journal' in journal_name and 'ssrn' not in journal_name:
            return 1
        typ = row['Work_Type']
        if typ == 'article': return 1
        if typ in ['book-chapter', 'book']: return 2
        return 3 

    df['Prio'] = df.apply(bestimme_prioritaet, axis=1)

    erhaltene_zeilen = []
    
    for title_group, group_df in df.groupby('Title_Group'):
        beste_prio = group_df['Prio'].min()
        beste_papers = group_df[group_df['Prio'] == beste_prio]
        beste_papers = beste_papers.sort_values(by='Datum', ascending=False)
        winner = beste_papers.iloc[[0]]
        erhaltene_zeilen.append(winner)

    df_dedup = pd.concat(erhaltene_zeilen, ignore_index=True)
    df_dedup = df_dedup.drop_duplicates(subset=['Link', 'Uni'], keep='first')

    return df_dedup.drop(columns=['Clean_Title', 'Title_Group', 'Prio']).sort_values(by=['Jahr', 'Datum']).reset_index(drop=True)

In [22]:
# --- HIER KANNST DU DIREKT TESTEN UND INSPEZIEREN ---
test_id = "A5027391165"
print("Lade und verarbeite Daten...")

raw_json = fetch_openalex_data(test_id)
df_raw = extract_flat_data(raw_json, test_id)

# HIER DEN NAMEN ÄNDERN: deduplicate_papers statt deduplicate_global!
df_clean = deduplicate_papers(df_raw) 

# Schau dir hier die flache Liste an. 
df_clean[['Jahr', 'Uni', 'Titel', 'Work_Type', 'Journal']]

Lade und verarbeite Daten...


,Jahr,Uni,Titel,Work_Type,Journal
0,2005,London School of Economics and Political Science,Does Tracking Aect the Importance of Family Background on Student's Test Scores?,article,Kein Journal angegeben
1,2010,University of Warwick,Studying Abroad and the Effect on International Labour Market Mobility: Evidence from the Introduction of ERASMUS,article,The Economic Journal
2,2010,University of Warwick,Quality Matters: The Expulsion of Professors and the Consequences for PhD Student Outcomes in Nazi Germany,article,Journal of Political Economy
3,2011,University of Warwick,German-Jewish Emigres and U.S. Invention,preprint,SSRN Electronic Journal
4,2011,University of Warwick,Peer Effects in Science: Evidence from the Dismissal of Scientists in Nazi Germany,article,The Review of Economic Studies
5,2015,University of Warwick,The Selection of High-Skilled Migrants,preprint,Econstor (Econstor)
6,2016,London School of Economics and Political Science,"Bombs, Brains, and Science: The Role of Human and Physical Capital for the Creation of Scientific Knowledge",article,The Review of Economics and Statistics
7,2017,London School of Economics and Political Science,acsanalysis.do,preprint,Harvard Dataverse
8,2017,London School of Economics and Political Science,acssetup.do,preprint,Harvard Dataverse
9,2017,London School of Economics and Political Science,selection_endog_5gr.ado,preprint,Harvard Dataverse
